In [1]:
import pandas as pd

# Charger le fichier
df = pd.read_json("data/testset.jsonl", lines=True)

# Afficher les 5 premières lignes
print(df.head())

# Pour tout voir dans ton IDE (VS Code / PyCharm)
# df.to_csv("check_questions.csv")

                                     id  \
0  201f2757-0dea-4d51-afe2-56344e6ed6bd   
1  9412e51c-1f13-437f-824b-f926f4ae11a1   
2  e023ce42-b65c-4163-a59f-796a028eff6e   
3  04e2773a-7456-479a-ad60-a3878846c491   
4  8e9f9ad9-fe65-4050-a4ec-5fc598762a3e   

                                            question  \
0  Quel titre a reçu l'article publié en 2023 par...   
1  Quels sont les auteurs de l'article "Creator: ...   
2  Quels sont les défis computationnels associés ...   
3  Quel est le nom du modèle qui démontre des ava...   
4  Quel est le titre de l'article de Lilian Some,...   

                                    reference_answer  \
0  A comprehensive survey on integrating large la...   
1  Cheng Qian, Chi Han, Y. Fung, Yujia Qin, Zhiyu...   
2  Ultra-long sequence context processing adresse...   
3                                        Qwen1.5 72B   
4  A comprehensive survey on integrating large la...   

                                   reference_context conversation_h

In [5]:
df

,id,question,reference_answer,reference_context,conversation_history,metadata
0,f2349c8e-cf54-4073-8efb-72e0ddbac642,Quels sont les modèles de langage ouverts que ...,"DeepSeek-V2 surpasse DeepSeek 67B, Qwen1.5 72B...",Document 44: Source: pdfs/2405.04434v5.pdf\nSe...,[],"{'question_type': 'simple', 'seed_document_id'..."
1,7dc2ed08-e864-4592-a794-d162cd292b9e,Quelle est la valeur de la fréquence d'apprent...,2.4 × 10^-4,Document 45: Source: pdfs/240504434v5.pdf\nSec...,[],"{'question_type': 'simple', 'seed_document_id'..."
2,abefb4d8-dd1b-4dec-b862-b4c4715891c4,Comment l'agent est-il évalué dans la tâche d'...,Nous comparons les ouvertures de conversation ...,Document 60: Source: pdfs/2310.08560v2.pdf\nSe...,[],"{'question_type': 'simple', 'seed_document_id'..."
3,dd75e1e9-ec27-4754-8a5f-e8e2e5aa06ad,Quel est le titre de l'article [1316] ?,Llm-based medical assistant personalization wi...,Document 70: Source: pdfs/2507.13334v2.pdf\nSe...,[],"{'question_type': 'simple', 'seed_document_id'..."
4,125373e9-06f5-4758-adb5-e0804e1df3b2,Quels sont les trois éléments intégraux de l'a...,Le système processe des nouvelles mémoires d'i...,Document 68: Source: pdfs/2502.12110v10.pdf\nS...,[],"{'question_type': 'simple', 'seed_document_id'..."
5,c4f074c9-cdcd-4027-91a8-f76de2cc216b,Quel est le titre de l'article écrit par T. Me...,Memory for multidimensional source information,Document 64: Source: pdfs/2507.13334v2.pdf\nSe...,[],"{'question_type': 'simple', 'seed_document_id'..."
6,2b77e096-5376-465a-9f5c-f2f7ef9bc570,Quels sont les contenus que Docling peut conve...,PDF documents,Document 56: Source: pdfs/2501.17887v1.pdf\nSe...,[],"{'question_type': 'simple', 'seed_document_id'..."
7,693cf3c1-e24d-40b4-8968-74d137730d26,Quel est l'architecture utilisée pour les FFNs ?,DeepSeekMoE,Document 49: Source: pdfs/2405.04434v5.pdf\nSe...,[],"{'question_type': 'simple', 'seed_document_id'..."
8,e24f6b8d-d6cd-4869-821f-91cf549a4fd7,Quel est le but du système A-MEM pour les agen...,A-MEM permet aux agents LLM d'avoir des capaci...,Document 54: Source: pdfs/2507.13334v2.pdf\nSe...,[],"{'question_type': 'simple', 'seed_document_id'..."
9,d2b2b8b2-6c71-48da-b5db-96ae8ddb9e37,DPO utilise-t-il l'apprentissage par renforcem...,"Non, DPO ne utilise pas l'apprentissage par re...",Document 53: Source: pdfs/1743034163900.pdf\nS...,[],"{'question_type': 'simple', 'seed_document_id'..."


# 1. Service

In [2]:
import os
import yaml
import logging
import asyncio
from typing import Any, Dict, List, Optional
from pathlib import Path
from dotenv import load_dotenv

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pymilvus import MilvusClient, AnnSearchRequest, RRFRanker
from FlagEmbedding import FlagReranker

# # --- 1. GESTION DES CHEMINS ---
# try:
#     # Si on est dans un script .py
#     CURRENT_DIR = Path(__file__).resolve().parent
# except NameError:
#     # Si on est dans un Notebook .ipynb
#     CURRENT_DIR = Path(os.getcwd())


import sys

# --- 1. GESTION DES CHEMINS ---
try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path(os.getcwd())

# On identifie le dossier src/ et la racine du projet
# Si le notebook est dans src/evaluation, CURRENT_DIR.parent donne src/
SRC_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "evaluation" else CURRENT_DIR
ROOT_DIR = SRC_DIR.parent

# Si le notebook est dans src/evaluation, on remonte de 2 niveaux pour arriver à la racine du projet
# (une fois pour sortir de evaluation, une fois pour sortir de src)
# Sinon, ajuste selon ta structure exacte.
if "evaluation" in str(CURRENT_DIR):
    PROJECT_ROOT = CURRENT_DIR.parent.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent

# AJOUT CRUCIAL : On ajoute la racine au chemin de recherche de Python
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    print(f"✅ Racine du projet ajoutée au sys.path : {PROJECT_ROOT}")

# Chargement du .env depuis la racine
load_dotenv(ROOT_DIR / ".env")


# Maintenant, l'import fonctionnera
from src.utils.custom_embedding import CustomEmbedder

# --- 2. CONFIGURATION ---
MILVUS_URI = os.getenv("MILVUS_URI", "http://localhost:19530")
COLL = os.getenv("MILVUS_COLLECTION")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = "qwen2.5:7b" #os.getenv("OLLAMA_MODEL", "llama3.2:3b")
RERANK_MODEL_NAME = os.getenv("RERANK_MODEL", "BAAI/bge-reranker-base")
HF_EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL_NAME")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("rag_service")

# --- 3. CLASSE RAGSERVICE ---
class RagService:
    def __init__(self):
        logger.info("🚀 Initialisation des composants lourds (Modèles)...")
        
        # Connexion Milvus
        self.milvus = MilvusClient(MILVUS_URI)
        
        # Embedder local (E5)
        self.embedder = CustomEmbedder(HF_EMBEDDING_MODEL)
        
        # Reranker LOCAL (Directement en mémoire pour la stabilité)
        logger.info(f"📦 Chargement du Reranker : {RERANK_MODEL_NAME}")
        self.reranker = FlagReranker(RERANK_MODEL_NAME, use_fp16=False)
        
        # LLM (Ollama)
        self.llm = ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_URL,
            temperature=0.0,
        )

        # Chargement des Prompts YAML
        self.prompts_path = SRC_DIR / "rag_server" / "prompt.yaml"
        if not self.prompts_path.exists():
            raise FileNotFoundError(f"❌ Fichier non trouvé : {self.prompts_path}")

        with open(self.prompts_path, 'r', encoding='utf-8') as f:
            self.prompts_data = yaml.safe_load(f)

        # Initialisation des chaînes LCEL
        self.rewriter_chain = self._setup_chain("query_rewriter_prompt")
        self.rag_chain = self._setup_chain("rag_template")

    def _setup_chain(self, template_name: str):
        data = self.prompts_data.get(template_name)
        if not data:
            raise ValueError(f"Template {template_name} absent du fichier YAML")
        
        messages = []
        # Gestion sécurisée des clés 'system' et 'human'
        system_content = data.get("system", "")
        if system_content:
            system_content = system_content.replace("/no_think", "").strip()
            messages.append(("system", system_content))
            
        human_content = data.get("human") or data.get("user") or data.get("question")
        if not human_content:
            raise KeyError(f"Clé 'human' manquante pour {template_name}")
            
        messages.append(("human", human_content))
        
        prompt = ChatPromptTemplate.from_messages(messages)
        return prompt | self.llm | StrOutputParser()

    def hybrid_retrieval(self, query: str, top_k: int) -> List[Dict]:
        """Recherche Hybride Milvus (Dense + Sparse)"""
        qvec = self.embedder(query)
        dense_req = AnnSearchRequest(
            data=[qvec], anns_field="vector",
            param={"metric_type": "COSINE", "params": {"ef": 100}},
            limit=top_k
        )
        sparse_req = AnnSearchRequest(
            data=[query], anns_field="sparse",
            param={"metric_type": "BM25"},
            limit=top_k
        )
        res = self.milvus.hybrid_search(
            COLL, [sparse_req, dense_req],
            RRFRanker(k=60), limit=top_k,
            output_fields=["id", "text", "source", "page_no", "section_title"]
        )
        return res[0] if res else []

    def local_rerank(self, query: str, hits: List[Any], top_k: int) -> List[Dict]:
        """Reranking BGE local"""
        if not hits: return []
        
        passages, meta = [], []
        for h in hits:
            ent = h.get('entity') if isinstance(h, dict) else h.entity
            text = ent.get("text", "")
            title = ent.get("section_title", "")
            
            passages.append(f"{title}\n\n{text}" if title else text)
            meta.append({
                "id": ent.get("id"), "text": text, "source": ent.get("source"),
                "page_no": ent.get("page_no"), "section_title": title
            })

        # Score BGE direct sur les paires (Question, Passage)
        scores = self.reranker.compute_score([[query, p[:2000]] for p in passages])
        
        for m, s in zip(meta, scores):
            m["rerank_score"] = float(s)
            
        meta.sort(key=lambda x: x["rerank_score"], reverse=True)
        return meta[:top_k]

    async def run_pipeline(self, query: str, chat_history: List[Dict] = None):
        """Pipeline RAG complet"""
        history = chat_history or []
        h_str = "\n".join([f"{m['role']}: {m['content']}" for m in history[-5:]]) or "(Vide)"

        # 1. Reformulation
        rewritten = self.rewriter_chain.invoke({"chat_history": h_str, "input": query})
        rewritten = rewritten.strip().split('\n')[0]

        # 2. Recherche (top 60) + Rerank (top 5)
        hits = self.hybrid_retrieval(rewritten, top_k=60)
        contexts = self.local_rerank(rewritten, hits, top_k=5)

        # 3. Génération finale avec contexte
        ctx_text = "\n\n".join([f"DOC {i+1}: {c['text']}" for i, c in enumerate(contexts)])
        answer = self.rag_chain.invoke({
            "context": ctx_text,
            "chat_history": h_str,
            "question": query
        })

        return {"answer": answer, "rewritten_query": rewritten, "contexts": contexts}

# --- 4. INITIALISATION (SINGLETON) ---
if 'service' not in globals():
    try:
        service = RagService()
        logger.info("✅ RagService prêt !")
    except Exception as e:
        logger.error(f"❌ Erreur d'initialisation : {e}")
else:
    logger.info("♻️ Service déjà en mémoire. Relance le noyau (Kernel) pour recharger les modèles.")

✅ Racine du projet ajoutée au sys.path : /Users/foulhad/repos/ministere_interieur


INFO:rag_service:🚀 Initialisation des composants lourds (Modèles)...
INFO:rag_service:📦 Chargement du Reranker : BAAI/bge-reranker-base


Mode TEI activé : Utilisation du serveur à l'adresse http://localhost:8083


INFO:rag_service:✅ RagService prêt !


In [4]:
class RagService:
    def __init__(self):
        logger.info("🚀 Initialisation du Pipeline Local Complet (Agentic RAG)...")
        
        # 1. Ressources
        self.milvus = MilvusClient(MILVUS_URI)
        self.embedder = CustomEmbedder(HF_EMBEDDING_MODEL)
        
        # Reranker Local (en mémoire)
        logger.info(f"📦 Chargement du Reranker : {RERANK_MODEL_NAME}")
        self.reranker = FlagReranker(RERANK_MODEL_NAME, use_fp16=False)
        
        # LLM (Ollama)
        self.llm = ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_URL,
            temperature=0.0,
        )

        # 2. Prompts
        self.prompts_path = SRC_DIR / "rag_server" / "prompt.yaml"
        with open(self.prompts_path, 'r', encoding='utf-8') as f:
            self.prompts_data = yaml.safe_load(f)

        # 3. Initialisation des Chaînes LCEL
        # Note : On utilise la fonction _setup_chain pour éviter la répétition
        self.rewriter_chain = self._setup_chain("query_rewriter_prompt")
        self.decomposition_chain = self._setup_chain("query_decomposition_multiquery_prompt")
        self.rag_chain = self._setup_chain("rag_template")
        self.relevance_chain = self._setup_chain("reflection_relevance_check_prompt")
        self.groundedness_chain = self._setup_chain("reflection_groundedness_check_prompt")
        self.regeneration_chain = self._setup_chain("reflection_response_regeneration_prompt")

    def _setup_chain(self, template_name: str):
        data = self.prompts_data.get(template_name)
        if not data: raise ValueError(f"Template {template_name} absent")
        
        messages = []
        if data.get("system"):
            messages.append(("system", data["system"].replace("/no_think", "").strip()))
        
        # Support pour les clés variables dans le YAML
        human_text = data.get("human") or data.get("user") or data.get("question")
        messages.append(("human", human_text))
        
        return ChatPromptTemplate.from_messages(messages) | self.llm | StrOutputParser()

    def hybrid_retrieval(self, query: str, top_k: int) -> List[Dict]:
        qvec = self.embedder(query)
        dense_req = AnnSearchRequest(data=[qvec], anns_field="vector", param={"metric_type": "COSINE", "params": {"ef": 100}}, limit=top_k)
        sparse_req = AnnSearchRequest(data=[query], anns_field="sparse", param={"metric_type": "BM25"}, limit=top_k)
        res = self.milvus.hybrid_search(COLL, [sparse_req, dense_req], RRFRanker(k=60), limit=top_k, output_fields=["id", "text", "source", "page_no", "section_title"])
        return res[0] if res else []

    def local_rerank(self, query: str, hits: List[Any], top_k: int) -> List[Dict]:
        if not hits: return []
        passages, meta = [], []
        for h in hits:
            ent = h.get('entity') if isinstance(h, dict) else h.entity
            text, title = ent.get("text", ""), ent.get("section_title", "")
            passages.append(f"{title}\n\n{text}" if title else text)
            meta.append({"id": ent.get("id"), "text": text, "source": ent.get("source"), "page_no": ent.get("page_no"), "section_title": title})

        # Scoring BGE Local
        scores = self.reranker.compute_score([[query, p[:2000]] for p in passages])
        for m, s in zip(meta, scores):
            m["rerank_score"] = float(s)
            
        # On garde le seuil de -8.0 pour la pertinence
        filtered = [m for m in meta if m["rerank_score"] >= -8.0]
        filtered.sort(key=lambda x: x["rerank_score"], reverse=True)
        return filtered[:top_k]

    async def run_pipeline(self, query: str, chat_history: List[Dict] = None):
        history = chat_history or []
        h_str = "\n".join([f"{m['role']}: {m['content']}" for m in history[-5:]]) or "(Vide)"

        # 1. Reformulation
        rewritten = self.rewriter_chain.invoke({"chat_history": h_str, "input": query})
        rewritten = rewritten.replace("Requête reformulée :", "").strip().split('\n')[0]

        # 2. Multi-Query (Décomposition)
        sub_queries_raw = self.decomposition_chain.invoke({"question": rewritten})
        sub_queries = [l.strip() for l in sub_queries_raw.split('\n') if l.strip() and any(c.isdigit() for c in l[:2])]
        if not sub_queries: sub_queries = [rewritten]

        # 3. Retrieval Global (On cherche sur toutes les sous-questions)
        all_hits = []
        for sq in sub_queries:
            all_hits.extend(self.hybrid_retrieval(sq, top_k=60))
        
        # Déduplication
        seen_ids, unique_hits = set(), []
        for h in all_hits:
            hid = h['id'] if isinstance(h, dict) else h.id
            if hid not in seen_ids:
                unique_hits.append(h); seen_ids.add(hid)

        # 4. Reranking Local
        contexts = self.local_rerank(rewritten, unique_hits, top_k=5)
        ctx_text = "\n\n".join([f"DOC {i+1}: {c['text']}" for i, c in enumerate(contexts)])

        # 5. Reflection: Relevance Check
        if contexts:
            rel_score = self.relevance_chain.invoke({"query": rewritten, "context": ctx_text})
            if "0" in rel_score:
                return {"answer": "Information non trouvée dans les documents.", "rewritten_query": rewritten, "contexts": []}

        # 6. Génération RAG
        answer = self.rag_chain.invoke({"context": ctx_text, "chat_history": h_str, "question": query})

        # 7. Reflection: Groundedness (Hallucination)
        ground_score = self.groundedness_chain.invoke({"context": ctx_text, "response": answer})
        if "0" in ground_score:
            logger.warning("🚨 Hallucination détectée, régénération...")
            answer = self.regeneration_chain.invoke({"context": ctx_text, "query": rewritten})

        return {"answer": answer, "rewritten_query": rewritten, "contexts": contexts}

# 1. Evaluate retrieval

In [5]:
import os
import sys
import pandas as pd
import numpy as np
import asyncio
from pathlib import Path
import nest_asyncio

nest_asyncio.apply()

# --- 1. Utilisation du Singleton service ---
# Le service est déjà initialisé dans la cellule précédente
if 'service' not in globals():
    from src.rag_server.rag_service import RagService
    service = RagService()

# --- 2. Fonction de matching simplifiée ---
def check_source_match(expected_source, final_contexts):
    """Vérifie si la source attendue est dans les résultats (Top K)."""
    if not expected_source:
        return None
    
    expected_clean = str(expected_source).strip().lower()
    
    for rank, ctx in enumerate(final_contexts, start=1):
        # Dans ton nouveau service, ctx est un dictionnaire
        current_src = str(ctx.get("source", "")).strip().lower()
        if expected_clean in current_src or current_src in expected_clean:
            return rank
    return None

# --- 3. Boucle d'évaluation robuste ---
async def run_eval_local(service, df_test, top_k_eval=5):
    results = []
    len_total = len(df_test)
    print(f"📊 Début de l'évaluation locale sur {len_total} questions...")

    for idx, (row_id, row) in enumerate(df_test.iterrows(), start=1):
        question = row["question"]
        # Giskard stocke souvent la source dans metadata ou reference_context
        # On adapte selon ton fichier testset_complex.jsonl
        ref_source = row.get("metadata", {}).get("source") or row.get("reference_context", "")

        try:
            # Utilisation du pipeline complet du service
            # (Rewriting + Hybrid Search + Local Rerank)
            output = await service.run_pipeline(query=str(question), chat_history=[])
            
            rewritten = output["rewritten_query"]
            final_contexts = output["contexts"] # Top K final (déjà reranké)

            # Scoring du Retrieval
            found_at_rank = check_source_match(ref_source, final_contexts)

            results.append({
                "question": question,
                "rewritten": rewritten,
                "expected_source": ref_source,
                "found_rank": found_at_rank,
                "hit": found_at_rank is not None,
                "answer": output["answer"][:100] + "..." # Pour vérification rapide
            })

        except Exception as e:
            print(f"❌ Erreur à l'index {idx} ({question[:30]}...): {e}")
            continue
        
        if idx % 5 == 0 or idx == len_total:
            print(f"✅ Traitées : {idx}/{len_total}")

    return pd.DataFrame(results)

# --- 4. Exécution ---
from giskard.rag import QATestset

# Chargement (ajuste le chemin si besoin)
testset_path = SRC_DIR / "evaluation" / "data" / "testset_complex.jsonl"
testset = QATestset.load(str(testset_path))
df_test = testset.to_pandas()

# Lancement
results_df = await run_eval_local(service, df_test)

# --- 5. Calcul des métriques ---
hit_rate = results_df["hit"].mean()
mrr = np.mean([1.0/r if pd.notnull(r) else 0 for r in results_df["found_rank"]])

print("\n" + "="*40)
print(f"🎯 RÉSULTATS FINAUX (Top {service.rag_chain.middle[0].top_k if hasattr(service, 'top_k_final') else 5})")
print(f"📈 Hit Rate (Précision) : {hit_rate:.2%}")
print(f"🔍 MRR (Qualité rang)   : {mrr:.3f}")
print("="*40)

display(results_df.head(10))

📊 Début de l'évaluation locale sur 7 questions...
✅ Traitées : 5/7
✅ Traitées : 7/7

🎯 RÉSULTATS FINAUX (Top 5)
📈 Hit Rate (Précision) : 100.00%
🔍 MRR (Qualité rang)   : 0.726


,question,rewritten,expected_source,found_rank,hit,answer
0,Quels sont les types de récompenses utilisés p...,Quels types de récompenses sont utilisés dans ...,Document 75: Source: pdfs/2507.13334v2.pdf\n\n...,4,True,Selon le document [1287] de Weizhe Yuan et al....
1,Quels sont les objectifs de la publication doc...,Quels sont les objectifs de l'utilisation du D...,Document 78: Source: pdfs/2501.17887v1.pdf\n\n...,1,True,"Les objectifs principaux de Docling, comme déc..."
2,Quels sont les formats de paiement que vous ac...,Quels sont les formats de paiement acceptés et...,Document 77: Source: pdfs/2507.13334v2.pdf\n\n...,3,True,"Désolé, mais votre question semble ne pas être..."
3,Quels mécanismes d'orchestration sont utilisés...,Quels mécanismes d'orchestration sont utilisés...,Document 13: Source: pdfs/2507.13334v2.pdf\n\n...,1,True,Le contexte fourni ne mentionne pas explicitem...
4,Quels sont les principes de l'ingénierie du co...,Quels sont les principes de l'ingénierie du co...,Document 89: Source: pdfs/2507.13334v2.pdf\n\n...,2,True,Le document fourni ne contient pas directement...
5,"Bonjour, je suis chercheur en université prest...",Quel est le but de la bibliothèque pypdf dans ...,Document 78: Source: pdfs/2501.17887v1.pdf\n\n...,1,True,La bibliothèque pypdf est une bibliothèque Pyt...
6,What are the microbatch size and learning rate...,What are the recommended microbatch size and l...,Document 68: Source: pdfs/2412.13663v2.pdf\n\n...,1,True,Pour la phase de pré-entraînement de DeepSeek-...


## 2. RAG Answer

In [12]:
!pip install ragas datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 41.1 MB/s  0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.12.0
    Uninstalling jiter-0.12.0:
      Successfully uninstalled jiter-0.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [ragas]32m5/6 [ragas]ctor]ork]r]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nv-ingest 25.9.0 requires scipy>=1.15.1, but you have scipy 1.11.4 which is incompatible.


In [6]:
import pandas as pd
import asyncio
from datasets import Dataset
from ragas import evaluate, RunConfig
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)
from ragas.llms import LangchainLLMWrapper
from langchain_ollama import ChatOllama

# --- 1. CONFIGURATION DU JUGE RAGAS ---
# Utilisation de Llama 3.1 8B (indispensable pour la fiabilité des scores)

from langchain_core.embeddings import Embeddings

# --- Wrapper pour rendre ton CustomEmbedder compatible avec Ragas ---
class RagasEmbeddingsWrapper(Embeddings):
    def __init__(self, custom_embedder):
        self.custom_embedder = custom_embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        # Ragas appelle ça pour les contextes
        return [self.custom_embedder(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        # Ragas appelle ça pour la question (C'est l'erreur que tu as eu !)
        return self.custom_embedder(text)

# --- Configuration du Juge corrigée ---
llm_juge = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0,
)

ragas_llm = LangchainLLMWrapper(llm_juge)

# On wrap ton embedder custom
ragas_embeddings = RagasEmbeddingsWrapper(service.embedder) 

print("✅ Juge Ragas configuré avec Wrapper d'embeddings compatible.")
llm_juge = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0,
)

# ragas_llm = LangchainLLMWrapper(llm_juge)
# # On utilise l'embedder déjà initialisé dans ton 'service'
# ragas_embeddings = service.embedder 

print("✅ Juge Ragas configuré sur Llama 3.1 8B")

# --- 2. FONCTION D'ÉVALUATION ---
async def run_ragas_eval_local(service, df_test):
    questions, answers, contexts, ground_truths = [], [], [], []

    print(f"🏃‍♂️ Phase 1 : Génération des réponses RAG ({len(df_test)} questions)...")

    for idx, (row_id, row) in enumerate(df_test.iterrows(), start=1):
        q = row["question"]
        try:
            # Appel au pipeline local (Reformulation + Milvus + Reranker + LLM)
            res = await service.run_pipeline(query=q, chat_history=[])
            
            questions.append(q)
            answers.append(res["answer"])
            # Ragas attend les textes bruts des documents récupérés
            contexts.append([c["text"] for c in res["contexts"]])
            # Réponse de référence pour le Recall
            ground_truths.append(row.get("reference_answer", ""))
            
            print(f"  - [{idx}/{len(df_test)}] Réponse générée avec succès")
        except Exception as e:
            print(f"  ⚠️ Erreur question {idx}: {e}")
            continue

    if not questions:
        print("❌ Erreur critique : Aucune réponse n'a été générée. Vérifiez la connexion à l'embedder TEI.")
        return None

    # Création du Dataset Ragas
    ds_dict = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    }
    dataset = Dataset.from_dict(ds_dict)

    # Configuration Anti-Timeout pour Ollama Local
    run_config = RunConfig(
        timeout=240,     # 4 min par évaluation
        max_workers=1    # STRICTEMENT 1 pour éviter la saturation CPU/GPU
    )

    print("\n⚖️  Phase 2 : Évaluation Ragas (Calcul des scores)...")
    metrics = [faithfulness, context_recall, context_precision]
    
    result = evaluate(
        dataset,
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=run_config
    )
    
    return result

# --- 3. EXÉCUTION ---
# On teste sur les 5 premières lignes du DataFrame chargé (df_test)
df_sample = df.head(5) 
results = await run_ragas_eval_local(service, df_sample)

# --- 4. AFFICHAGE ET ANALYSE ---
print("\n📊 SYNTHÈSE DES SCORES RAGAS :")
df_results = results.to_pandas()

# Détection dynamique des colonnes pour éviter le KeyError "question"
all_cols = df_results.columns.tolist()
# Ragas utilise parfois 'user_input' au lieu de 'question' selon la version
input_col = [c for c in ["question", "user_input"] if c in all_cols]
score_cols = [c for c in ["faithfulness", "context_recall", "context_precision"] if c in all_cols]

# Affichage final
display(df_results[input_col + score_cols])

# Sauvegarde pour archivage
df_results.to_csv("evaluation_ragas_ministere.csv", index=False)
print("\n💾 Résultats sauvegardés dans 'evaluation_ragas_ministere.csv'")

/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn/T/ipykernel_1444/1823511136.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn/T/ipykernel_1444/1823511136.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn/T/ipykernel_1444/1823511136.py:5: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/var/folde

✅ Juge Ragas configuré avec Wrapper d'embeddings compatible.
✅ Juge Ragas configuré sur Llama 3.1 8B
🏃‍♂️ Phase 1 : Génération des réponses RAG (5 questions)...
  - [1/5] Réponse générée avec succès
  - [2/5] Réponse générée avec succès
  - [3/5] Réponse générée avec succès
  - [4/5] Réponse générée avec succès
  - [5/5] Réponse générée avec succès

⚖️  Phase 2 : Évaluation Ragas (Calcul des scores)...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]


📊 SYNTHÈSE DES SCORES RAGAS :


,user_input,faithfulness,context_recall,context_precision
0,Quel titre a reçu l'article publié en 2023 par...,0.000000,0.166667,1.000000
1,"Quels sont les auteurs de l'article ""Creator: ...",1.000000,1.000000,1.000000
2,Quels sont les défis computationnels associés ...,1.000000,1.000000,0.887500
3,Quel est le nom du modèle qui démontre des ava...,0.333333,1.000000,0.866667
4,"Quel est le titre de l'article de Lilian Some,...",0.250000,1.000000,1.000000



💾 Résultats sauvegardés dans 'evaluation_ragas_ministere.csv'
